# Unified Geoscience Model — Cloud Training Notebook

**Murmurative** slot-attention architecture for geoscience AI.

Four-phase progressive curriculum:
1. **Phase 1** — Encoder pretraining (masked auto-encoding on synthetic data)
2. **Phase 2** — Task head training (facies, lithology, kriging)
3. **Phase 3** — Language model (MoE decoder on geology text)
4. **Phase 4** — Joint finetuning (everything together, reduced LR)

Runs on GPU (T4/V100/A100) — no local hardware needed.

## 0. Environment Setup

In [ ]:
import sys, os, subprocess, math, time, json
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/tostal.git"
REPO_DIR = Path("/kaggle/working/tostal") if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else Path.cwd() / "tostal"

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

os.chdir(REPO_DIR / "training")
CHECKPOINT_DIR = (Path("/kaggle/working") if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else Path.cwd()) / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working dir: {os.getcwd()}")
print(f"Checkpoints → {CHECKPOINT_DIR}")

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>/dev/null
!pip install -q numpy pyyaml tqdm matplotlib

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm, trange

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  Device: {DEVICE}")
if DEVICE == "cuda":
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Import Model & Configure Training

In [ ]:
sys.path.insert(0, str(Path.cwd() / "src"))

from unified_geo.model import UnifiedGeoscienceModel
from unified_geo.config import ModelConfig, TrainingConfig
from unified_geo.data import mixed_batch_iterator, geology_text_generator

model_cfg = ModelConfig(
    d_model=512,
    num_heads=8,
    num_slots=512,
    num_rounds=3,
    alpha=0.9,
    gamma=0.15,
    k_neighbors=7,
    vocab_size=32000,
    num_experts=6,
    experts_per_token=2,
    decoder_layers=6,
    facies_num_classes=12,
    lithology_num_classes=12,
)

train_cfg = TrainingConfig(
    lr=5e-4,
    weight_decay=0.01,
    warmup_steps=500,
    max_steps=10000,
    batch_size=8,
    grad_clip=1.0,
    phase2_freeze_steps=2500,
    phase4_lr_multiplier=0.1,
)

model = UnifiedGeoscienceModel(model_cfg).to(DEVICE)

param_counts = model.count_params()
print(f"Total parameters: {param_counts['total']:,}")
for k, v in param_counts.items():
    if k != "total":
        print(f"  {k}: {v:,}")

## Utility functions

In [ ]:
def create_optimizer(model, lr, weight_decay=0.01):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, eps=1e-4)

def cosine_lr_schedule(optimizer, step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        lr = base_lr * step / max(1, warmup_steps)
    else:
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        lr = base_lr * 0.5 * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr

def freeze_module(m):
    for p in m.parameters():
        p.requires_grad = False

def unfreeze_module(m):
    for p in m.parameters():
        p.requires_grad = True

def save_checkpoint(model, optimizer, step, loss_history, path, metadata=None):
    path = str(path)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "step": step,
        "loss_history": loss_history,
        "metadata": metadata or {},
    }, path)

def load_checkpoint(path, map_location=DEVICE):
    data = torch.load(path, map_location=map_location, weights_only=False)
    return data

loss_history = {}

In [ ]:
def plot_losses(phase_name, losses):
    plt.figure(figsize=(12, 3))
    plt.plot(losses, alpha=0.3, color="#4C72B0", linewidth=0.5)
    window = max(1, len(losses) // 40)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    plt.plot(np.arange(window-1, len(losses)), smoothed, color="#C44E52", linewidth=1.5)
    plt.title(f"{phase_name} — Training Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

---
## Phase 1 — Encoder Pretraining

Masked auto-encoding on synthetic well logs, images, and spatial data. Only the encoder is trained.

In [ ]:
print("=== Phase 1: Encoder Pretraining ===\n")

model.train()
optimizer = create_optimizer(model.encoder, train_cfg.lr, train_cfg.weight_decay)
phase1_losses = []

data_iter = mixed_batch_iterator(
    train_cfg.batch_size, model_cfg, train_cfg, DEVICE,
    num_batches=train_cfg.max_steps,
)

pbar = trange(train_cfg.max_steps, desc="Phase 1", unit="step")
for step in pbar:
    batch = next(data_iter)
    lr = cosine_lr_schedule(optimizer, step, train_cfg.max_steps, train_cfg.warmup_steps, train_cfg.lr)

    results = model(well_log=batch["well_log"], image=batch["image"], spatial=batch["spatial"], mode="encode")

    well_pred  = model.encoder.well_log_encoder(batch["well_log"])
    well_loss  = F.mse_loss(well_pred, model.encoder.well_log_encoder(batch["well_target"]))
    image_pred = model.encoder.image_encoder(batch["image"])
    image_loss = F.mse_loss(image_pred, model.encoder.image_encoder(batch["image_target"]))
    spatial_vals = batch["spatial_target"][..., 3:]
    spatial_loss = F.mse_loss(
        results["encoder_tokens"][:, :batch["spatial_target"].shape[1], :3].mean(-1),
        spatial_vals.squeeze(-1),
    )

    loss = well_loss + image_loss + spatial_loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
    optimizer.step()

    phase1_losses.append(loss.item())
    if (step + 1) % 100 == 0:
        avg = sum(phase1_losses[-100:]) / 100
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{math.exp(min(avg,20)):.2f}")

pbar.close()
loss_history["phase1"] = phase1_losses

save_checkpoint(model, optimizer, train_cfg.max_steps, phase1_losses,
                CHECKPOINT_DIR / "phase1" / "model.pt", {"phase": 1})

print(f"\nPhase 1 complete — final loss: {sum(phase1_losses[-100:])/100:.4f}")
plot_losses("Phase 1 — Encoder Pretraining", phase1_losses)

---
## Phase 2 — Task Head Training

Freeze encoder → train facies/lithology/kriging heads. After 2,500 steps unfreeze encoder for joint fine-tuning.

In [ ]:
print("=== Phase 2: Task Head Training ===\n")

model.train()
freeze_module(model.encoder)
optimizer = create_optimizer(model, train_cfg.lr, train_cfg.weight_decay)
phase2_losses = []

data_iter = mixed_batch_iterator(
    train_cfg.batch_size, model_cfg, train_cfg, DEVICE,
    num_batches=train_cfg.max_steps,
)

pbar = trange(train_cfg.max_steps, desc="Phase 2", unit="step")
for step in pbar:
    if step == train_cfg.phase2_freeze_steps:
        unfreeze_module(model.encoder)
        pbar.write(f"  ▸ Unfroze encoder at step {step}")

    batch = next(data_iter)
    lr = cosine_lr_schedule(optimizer, step, train_cfg.max_steps, train_cfg.warmup_steps, train_cfg.lr)

    results = model(well_log=batch["well_log"], image=batch["image"], spatial=batch["spatial"], mode="encode")

    facies_target = torch.randint(0, model_cfg.facies_num_classes, (train_cfg.batch_size,), device=DEVICE)
    facies_loss = F.cross_entropy(results["facies_logits"], facies_target)

    litho_target = torch.randint(0, model_cfg.lithology_num_classes, (train_cfg.batch_size, 64, 64), device=DEVICE)
    litho_loss = F.cross_entropy(results["lithology_logits"], litho_target)

    krige_q = torch.rand(train_cfg.batch_size, 32, 3, device=DEVICE)
    krige_target = torch.rand(train_cfg.batch_size, 32, device=DEVICE)
    krige_pred = model.kriging_head(results["slot_v"], krige_q)
    krige_loss = F.mse_loss(krige_pred, krige_target)

    loss = facies_loss + litho_loss + krige_loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
    optimizer.step()

    phase2_losses.append(loss.item())
    if (step + 1) % 100 == 0:
        avg = sum(phase2_losses[-100:]) / 100
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{math.exp(min(avg,20)):.2f}")

pbar.close()
loss_history["phase2"] = phase2_losses

save_checkpoint(model, optimizer, train_cfg.max_steps, phase2_losses,
                CHECKPOINT_DIR / "phase2" / "model.pt", {"phase": 2})

print(f"\nPhase 2 complete — final loss: {sum(phase2_losses[-100:])/100:.4f}")
plot_losses("Phase 2 — Task Head Training", phase2_losses)

---
## Phase 3 — Language Model Training

Freeze encoder + task heads → train only the MoE decoder as a geology language model.

In [ ]:
print("=== Phase 3: Language Model Training ===\n")

model.train()
freeze_module(model.encoder)
freeze_module(model.facies_head)
freeze_module(model.lithology_head)
freeze_module(model.kriging_head)
optimizer = create_optimizer(model.decoder, train_cfg.lr, train_cfg.weight_decay)
phase3_losses = []

data_iter = mixed_batch_iterator(
    train_cfg.batch_size, model_cfg, train_cfg, DEVICE,
    num_batches=train_cfg.max_steps,
)
text_iter = geology_text_generator(
    model_cfg.vocab_size, train_cfg.text_seq_len, train_cfg.max_steps, seed=42,
)

pbar = trange(train_cfg.max_steps, desc="Phase 3", unit="step")
for step in pbar:
    batch = next(data_iter)
    text_batch = next(text_iter).expand(train_cfg.batch_size, -1).to(DEVICE)

    lr = cosine_lr_schedule(optimizer, step, train_cfg.max_steps, train_cfg.warmup_steps, train_cfg.lr)

    results = model(
        well_log=batch["well_log"], image=batch["image"], spatial=batch["spatial"],
        decoder_input_ids=text_batch[:, :-1], mode="decode",
    )

    lm_loss = F.cross_entropy(results["lm_logits"].flatten(0, 1), text_batch[:, 1:].flatten(0, 1))
    loss = lm_loss + train_cfg.load_balance_alpha * results["balance_loss"] + train_cfg.router_z_loss_beta * results["z_loss"]

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
    optimizer.step()

    phase3_losses.append(lm_loss.item())
    if (step + 1) % 100 == 0:
        avg = sum(phase3_losses[-100:]) / 100
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{math.exp(min(avg,20)):.2f}")

pbar.close()
loss_history["phase3"] = phase3_losses

save_checkpoint(model, optimizer, train_cfg.max_steps, phase3_losses,
                CHECKPOINT_DIR / "phase3" / "model.pt", {"phase": 3})

print(f"\nPhase 3 complete — final loss: {sum(phase3_losses[-100:])/100:.4f}")
plot_losses("Phase 3 — Language Model Training", phase3_losses)

---
## Phase 4 — Joint Finetuning

Unfreeze everything → joint training on all modalities + task heads + language model with 0.1x learning rate.

In [ ]:
print("=== Phase 4: Joint Finetuning ===\n")

model.train()
unfreeze_module(model.encoder)
unfreeze_module(model.facies_head)
unfreeze_module(model.lithology_head)
unfreeze_module(model.kriging_head)
unfreeze_module(model.decoder)

base_lr = train_cfg.lr * train_cfg.phase4_lr_multiplier
optimizer = create_optimizer(model, base_lr, train_cfg.weight_decay)
phase4_losses = []

data_iter = mixed_batch_iterator(
    train_cfg.batch_size, model_cfg, train_cfg, DEVICE,
    num_batches=train_cfg.max_steps,
)
text_iter = geology_text_generator(
    model_cfg.vocab_size, train_cfg.text_seq_len, train_cfg.max_steps, seed=99,
)

pbar = trange(train_cfg.max_steps, desc="Phase 4", unit="step")
for step in pbar:
    batch = next(data_iter)
    text_batch = next(text_iter).expand(train_cfg.batch_size, -1).to(DEVICE)

    lr = cosine_lr_schedule(optimizer, step, train_cfg.max_steps, train_cfg.warmup_steps, base_lr)

    results = model(
        well_log=batch["well_log"], image=batch["image"], spatial=batch["spatial"],
        decoder_input_ids=text_batch[:, :-1],
        kriging_query_points=torch.rand(train_cfg.batch_size, 32, 3, device=DEVICE),
        mode="full",
    )

    well_pred  = model.encoder.well_log_encoder(batch["well_log"])
    well_loss  = F.mse_loss(well_pred, model.encoder.well_log_encoder(batch["well_target"]))
    image_pred = model.encoder.image_encoder(batch["image"])
    image_loss = F.mse_loss(image_pred, model.encoder.image_encoder(batch["image_target"]))

    facies_target = torch.randint(0, model_cfg.facies_num_classes, (train_cfg.batch_size,), device=DEVICE)
    facies_loss = F.cross_entropy(results["facies_logits"], facies_target)

    litho_target = torch.randint(0, model_cfg.lithology_num_classes, (train_cfg.batch_size, 64, 64), device=DEVICE)
    litho_loss = F.cross_entropy(results["lithology_logits"], litho_target)

    lm_loss = F.cross_entropy(results["lm_logits"].flatten(0, 1), text_batch[:, 1:].flatten(0, 1))

    loss = (
        well_loss + image_loss + facies_loss + litho_loss + lm_loss +
        train_cfg.load_balance_alpha * results["balance_loss"] +
        train_cfg.router_z_loss_beta * results["z_loss"]
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
    optimizer.step()

    phase4_losses.append(loss.item())
    if (step + 1) % 100 == 0:
        avg = sum(phase4_losses[-100:]) / 100
        pbar.set_postfix(loss=f"{avg:.4f}", ppl=f"{math.exp(min(avg,20)):.2f}")

pbar.close()
loss_history["phase4"] = phase4_losses

save_checkpoint(model, optimizer, train_cfg.max_steps, phase4_losses,
                CHECKPOINT_DIR / "phase4" / "model.pt", {"phase": 4})

print(f"\nPhase 4 complete — final loss: {sum(phase4_losses[-100:])/100:.4f}")
plot_losses("Phase 4 — Joint Finetuning", phase4_losses)

---
## Summary — All Phases

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
for ax, (name, losses), c in zip(axes.flat, loss_history.items(), colors):
    ax.plot(losses, alpha=0.2, color=c, linewidth=0.5)
    w = max(1, len(losses) // 40)
    smoothed = np.convolve(losses, np.ones(w)/w, mode="valid")
    ax.plot(range(w-1, len(losses)), smoothed, color=c, linewidth=2)
    ax.set_title(name.replace("_", " ").title())
    ax.set_xlabel("Step"); ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.15)
fig.suptitle("Unified Geoscience Model — 4-Phase Training", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "training_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
for name, losses in loss_history.items():
    final = sum(losses[-100:]) / 100
    ppl = math.exp(min(final, 20))
    print(f"  {name:>8s}  →  loss={final:.4f}  ppl={ppl:.2f}")
print(f"\nCheckpoints saved to: {CHECKPOINT_DIR}")
print("\nTo export for inference:")
print(f"  torch.save({{'state_dict': model.state_dict(), 'config': {{...}}}}, '{CHECKPOINT_DIR / 'final_model.pt'}')")

In [ ]:
model.save(str(CHECKPOINT_DIR / "final_model.pt"))
print(f"Final model saved → {CHECKPOINT_DIR / 'final_model.pt'}")
print(f"Size: {os.path.getsize(CHECKPOINT_DIR / 'final_model.pt') / 1e6:.1f} MB")